# Notebook 03 — Matrix Multiplication, 2D Convolution, and Prefix Sums

**Session:** List for 2D data  
**Module:** 02 · Calculus and Linear Algebra

## Goal

Implement pure-Python matrix multiplication, zero-padded 2D box filtering (cross-correlation),
and $\mathcal{O}(1)$ rectangular queries via 2D prefix sums (integral images), then compare
with NumPy.

In [ ]:
import json
from pathlib import Path

import numpy as np

with Path("../data/matrix_benchmarks.json").open(encoding="utf-8") as fp:
    benchmarks = json.load(fp)

## 1. Pure-Python Matrix Multiplication

In [ ]:
def matmul(a: list[list[float]], b: list[list[float]]) -> list[list[float]]:
    """Compute matrix product A @ B for compatible 2D lists."""
    m, k_a = len(a), len(a[0])
    k_b, n = len(b), len(b[0])
    if k_a != k_b:
        raise ValueError(f"Incompatible shapes ({m}, {k_a}) and ({k_b}, {n}).")

    c = [[0.0] * n for _ in range(m)]
    for i in range(m):
        for k in range(k_a):
            a_ik = a[i][k]
            for j in range(n):
                c[i][j] += a_ik * b[k][j]
    return c


prod = matmul(benchmarks["matrix_a"], benchmarks["matrix_b"])
assert prod == benchmarks["expected_product_ab"]
assert np.allclose(prod, np.array(benchmarks["matrix_a"]) @ np.array(benchmarks["matrix_b"]))
print("Matrix product A @ B:", prod)

## 2. Zero-Padding and 2D Discrete Cross-Correlation

In [ ]:
def zero_pad_2d(image: list[list[float]], pad: int) -> list[list[float]]:
    """Pad a 2D matrix with *pad* layers of zeros on all four sides."""
    w = len(image[0])
    out_w = w + 2 * pad
    padded = [[0.0] * out_w for _ in range(pad)]
    for row in image:
        padded.append([0.0] * pad + list(row) + [0.0] * pad)
    padded.extend([[0.0] * out_w for _ in range(pad)])
    return padded


def correlate_2d(
    image: list[list[float]],
    kernel: list[list[float]],
    pad: int = 0,
) -> list[list[float]]:
    """Apply valid 2D cross-correlation after optional zero-padding."""
    src = zero_pad_2d(image, pad) if pad > 0 else image
    h, w = len(src), len(src[0])
    kh, kw = len(kernel), len(kernel[0])
    out_h, out_w = h - kh + 1, w - kw + 1

    output = [[0.0] * out_w for _ in range(out_h)]
    for i in range(out_h):
        for j in range(out_w):
            acc = 0.0
            for u in range(kh):
                for v in range(kw):
                    acc += src[i + u][j + v] * kernel[u][v]
            output[i][j] = acc
    return output


patch = benchmarks["image_patch_5x5"]
box_kernel = [[val / 9.0 for val in row] for row in benchmarks["box_kernel_3x3"]]
smoothed_valid = correlate_2d(patch, box_kernel, pad=0)
print("Valid 3x3 box-filtered 5x5 patch (3x3 output):")
for row in smoothed_valid:
    print(" ", [round(x, 2) for x in row])

## 3. 2D Prefix Sums (Integral Image) for O(1) Patch Queries

In [ ]:
def build_prefix_sum_2d(grid: list[list[float]]) -> list[list[float]]:
    """Build a (H+1) x (W+1) 2D prefix-sum table."""
    h, w = len(grid), len(grid[0])
    prefix = [[0.0] * (w + 1) for _ in range(h + 1)]
    for i in range(1, h + 1):
        for j in range(1, w + 1):
            prefix[i][j] = (
                grid[i - 1][j - 1] + prefix[i - 1][j] + prefix[i][j - 1] - prefix[i - 1][j - 1]
            )
    return prefix


def query_rect_sum(
    prefix: list[list[float]],
    r1: int,
    c1: int,
    r2: int,
    c2: int,
) -> float:
    """Return sum over inclusive submatrix [r1..r2, c1..c2] in O(1) time."""
    return prefix[r2 + 1][c2 + 1] - prefix[r1][c2 + 1] - prefix[r2 + 1][c1] + prefix[r1][c1]


prefix = build_prefix_sum_2d(patch)
box_sum_kernel = benchmarks["box_kernel_3x3"]
unnorm = correlate_2d(patch, box_sum_kernel, pad=0)

for i in range(3):
    for j in range(3):
        fast_sum = query_rect_sum(prefix, i, j, i + 2, j + 2)
        assert abs(fast_sum - unnorm[i][j]) < 1e-9

print("All 3x3 window sums via O(1) integral image match brute-force 2D correlation!")